In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt```

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
boto3==1.24.59
shap==0.40.0
matplotlib==3.6.1
catboost==1.0.4
seaborn==0.11.2

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import catboost as cb
import sklearn.metrics as skm
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import pickle

# get roc auc score (helper)
def get_roc_auc(y_true, y_score):
    flt_metric = skm.roc_auc_score(
        y_true=y_true, 
        y_score=y_score,
    )
    return flt_metric

# create binary target for age
def create_binary_target(val_target, val_threshold=60):
    if val_target < val_threshold:
        return 0
    elif val_target >= val_threshold:
        return 1
    else:
        return np.nan

# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)
    
# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

# constants
str_project = '20231010-gen-xii'
str_target = 'target'
str_dirname_output = './output'

# make output dir
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

###############################################################################
# HYPERPARAMETERS
###############################################################################
# get df_hyperparameters
print('Getting hyperparameters...')
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/12_step_function/{str_filename}'
df = pd.read_csv(str_uri)
# convert to dict
dict_hyperparameters = dict(zip(df['keys'], df['values']))

# get filename for train
str_filename_train = dict_hyperparameters['STR_FILENAME_TRAIN']
print(f'Train filename: {str_filename_train}')

# get filename for valid
str_filename_valid = dict_hyperparameters['STR_FILENAME_VALID']
print(f'Valid filename: {str_filename_valid}')

###############################################################################
# MODEL
###############################################################################
# import model
print('Importing best model...')
str_filename = 'df_tuning.csv'
str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
df = pd.read_csv(str_uri)
# get iteration
int_best_iteration = df['iteration'].iloc[0]

# get model
str_filename = f'dict_model_inference_{int_best_iteration}.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'02_pricing_pd/02_model/02_model/02_batch_tuning/models/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
dict_pipeline = pickle.load(open(str_local_path, 'rb'))
cls_model_inference = dict_pipeline['model_inference']
list_cols_model = list(cls_model_inference.feature_names_)
list_cols_import = list_cols_model + ['uniqueid', str_target]

try:
    ###############################################################################
    # RACE
    ###############################################################################
    # import valid data
    print('Race...')
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df_valid = pd.read_parquet(str_uri, columns=list_cols_import)

    # get the race
    str_filename = 'df_race_gender_age.csv'
    str_uri = f's3://{str_project}/ad_hoc/compliance/{str_filename}'
    df_tmp = pd.read_csv(str_uri, usecols=['uniqueid', 'max_proba_race', 'race'])
    df_tmp.dropna(subset=['race'], inplace=True)

    # join
    df_valid = pd.merge(
        left=df_valid,
        right=df_tmp,
        on='uniqueid',
        how='inner',
    )

    # subset
    df_valid = df_valid[df_valid['max_proba_race'] >= 0.70] # 0.7 is the threshold we are using for race certainty

    # get predictions
    df_valid['yhat'] = cls_model_inference.predict_proba(df_valid[list_cols_model])[:,1]

    # group
    list_cols = [
        'race',
        str_target,
        'yhat',
    ]
    df_grouped = df_valid[list_cols].groupby(by='race', as_index=False).agg({
        str_target: lambda x: list(x),
        'yhat': lambda x: list(x),
    })

    # make plot
    fig, ax = plt.subplots(figsize=(9,5))
    ax.set_title('Distributions of Predictions by Race in Validation Data')
    for a in range(df_grouped.shape[0]):
        str_label = f"{df_grouped['race'].iloc[a]}"
        sns.kdeplot(
            df_grouped['yhat'].iloc[a],
            ax=ax,
            label=str_label,
        )
    plt.legend()
    plt.savefig(f'{str_dirname_output}/plt_yhat_dist_race.png', bbox_inches='tight')

    # save memory
    del df_grouped

    # build model
    print('Building model to predict race...')
    # get training data
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
    df_train = pd.read_parquet(str_uri, columns=list_cols_import)

    # join
    df_train = pd.merge(
        left=df_train,
        right=df_tmp,
        on='uniqueid',
        how='inner',
    )

    # make mapping dictionary
    dict_map = {
        'white': 0,
        'black': 1,
        'hispanic': 2,
        'native': 3,
        'api': 4,
        'multiple': 5,
    }
    # map race
    df_train['race'] = df_train['race'].map(dict_map)
    df_valid['race'] = df_valid['race'].map(dict_map)

    # get non-numeric
    list_non_numeric = []
    for col in list_cols_model:
        if df_train[col].dtype not in ['float64','int64']:
            list_non_numeric.append(col)

    # pool train
    pool_train = cb.Pool(
        df_train[list_cols_model],
        df_train['race'],
        list_non_numeric,
    )

    # create class weights
    list_class_weights = list(1 - df_train['race'].value_counts(normalize=True))

    del df_train

    # pool valid
    pool_valid = cb.Pool(
        df_valid[list_cols_model],
        df_valid['race'],
        list_non_numeric,
    )
    del df_valid

    # init
    cls_model_inference = cb.CatBoostClassifier(
        task_type='CPU',
        nan_mode='Min',
        random_state=42,
        eval_metric='MultiClass',
        iterations=100,
        learning_rate=None, # default
        class_weights=list_class_weights,
    )

    # fit
    cls_model_inference.fit(
        pool_train,
        eval_set=[pool_valid],
        verbose=10,
        use_best_model=True,
        early_stopping_rounds=10,
    )

    # get feat imp
    df_feat_imp = pd.DataFrame({
        'feature': cls_model_inference.feature_names_,
        'importance': cls_model_inference.feature_importances_,
    })
    df_feat_imp.sort_values(by='importance', ascending=False, inplace=True)
    # write to s3
    str_filename = 'df_feat_imp_race.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/11b_disparate_impact/{str_filename}'
    df_feat_imp.to_csv(str_uri, index=False)

    ###############################################################################
    # GENDER
    ###############################################################################
    # import valid data
    print('Gender...')
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df_valid = pd.read_parquet(str_uri, columns=list_cols_import)

    # get the gender
    str_filename = 'df_race_gender_age.csv'
    str_uri = f's3://{str_project}/ad_hoc/compliance/{str_filename}'
    df_tmp = pd.read_csv(str_uri, usecols=['uniqueid', 'gender'])
    df_tmp.dropna(subset=['gender'], inplace=True)

    # join
    df_valid = pd.merge(
        left=df_valid,
        right=df_tmp,
        on='uniqueid',
        how='inner',
    )

    # get predictions
    df_valid['yhat'] = cls_model_inference.predict_proba(df_valid[list_cols_model])[:,1]

    # group
    list_cols = [
        'gender',
        str_target,
        'yhat',
    ]
    df_grouped = df_valid[list_cols].groupby(by='gender', as_index=False).agg({
        str_target: lambda x: list(x),
        'yhat': lambda x: list(x),
    })

    # make plot
    fig, ax = plt.subplots(figsize=(9,5))
    ax.set_title('Distributions of Predictions by Gender in Validation Data')
    for a in range(df_grouped.shape[0]):
        str_label = f"{df_grouped['gender'].iloc[a]}"
        sns.kdeplot(
            df_grouped['yhat'].iloc[a],
            ax=ax,
            label=str_label,
        )
    plt.legend()
    plt.savefig(f'{str_dirname_output}/plt_yhat_dist_gender.png', bbox_inches='tight')

    # save memory
    del df_grouped

    # build model
    print('Building model to predict gender...')
    # get training data
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
    df_train = pd.read_parquet(str_uri, columns=list_cols_import)

    # join
    df_train = pd.merge(
        left=df_train,
        right=df_tmp,
        on='uniqueid',
        how='inner',
    )

    # make mapping dictionary
    dict_map = {
        'male': 0,
        'female': 1,
    }
    # map gender
    df_train['gender'] = df_train['gender'].map(dict_map)
    df_valid['gender'] = df_valid['gender'].map(dict_map)

    # get non-numeric
    list_non_numeric = []
    for col in list_cols_model:
        if df_train[col].dtype not in ['float64','int64']:
            list_non_numeric.append(col)

    # pool train
    pool_train = cb.Pool(
        df_train[list_cols_model],
        df_train['gender'],
        list_non_numeric,
    )

    del df_train

    # pool valid
    pool_valid = cb.Pool(
        df_valid[list_cols_model],
        df_valid['gender'],
        list_non_numeric,
    )
    del df_valid

    # init
    cls_model_inference = cb.CatBoostClassifier(
        task_type='CPU',
        nan_mode='Min',
        random_state=42,
        eval_metric='AUC',
        iterations=100,
        learning_rate=None, # default
    )

    # fit
    cls_model_inference.fit(
        pool_train,
        eval_set=[pool_valid],
        verbose=10,
        use_best_model=True,
        early_stopping_rounds=10,
    )

    # get feat imp
    df_feat_imp = pd.DataFrame({
        'feature': cls_model_inference.feature_names_,
        'importance': cls_model_inference.feature_importances_,
    })
    df_feat_imp.sort_values(by='importance', ascending=False, inplace=True)
    # write to s3
    str_filename = 'df_feat_imp_gender.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/11b_disparate_impact/{str_filename}'
    df_feat_imp.to_csv(str_uri, index=False)

    ###############################################################################
    # AGE
    ###############################################################################
    # import valid data
    print('Age...')
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df_valid = pd.read_parquet(str_uri, columns=list_cols_import)

    # get the gender
    str_filename = 'df_race_gender_age.csv'
    str_uri = f's3://{str_project}/ad_hoc/compliance/{str_filename}'
    df_tmp = pd.read_csv(str_uri, usecols=['uniqueid', 'subjectage__ln'])
    df_tmp.dropna(subset=['subjectage__ln'], inplace=True)
    # make binary
    df_tmp['subjectage__ln'] = df_tmp['subjectage__ln'].apply(create_binary_target)

    # join
    df_valid = pd.merge(
        left=df_valid,
        right=df_tmp,
        on='uniqueid',
        how='inner',
    )

    # get predictions
    df_valid['yhat'] = cls_model_inference.predict_proba(df_valid[list_cols_model])[:,1]

    # group
    list_cols = [
        'subjectage__ln',
        str_target,
        'yhat',
    ]
    df_grouped = df_valid[list_cols].groupby(by='subjectage__ln', as_index=False).agg({
        str_target: lambda x: list(x),
        'yhat': lambda x: list(x),
    })

    # make plot
    fig, ax = plt.subplots(figsize=(9,5))
    ax.set_title('Distributions of Predictions by Age in Validation Data')
    for a in range(df_grouped.shape[0]):
        int_age_group = df_grouped['subjectage__ln'].iloc[a]
        # logic
        if int_age_group == 0:
            str_label = 'Under 60'
        else:
            str_label = '60 +'
        sns.kdeplot(
            df_grouped['yhat'].iloc[a],
            ax=ax,
            label=str_label,
        )
    plt.legend()
    plt.savefig(f'{str_dirname_output}/plt_yhat_dist_age.png', bbox_inches='tight')

    # save memory
    del df_grouped

    # build model
    print('Building model to predict age...')
    # get training data
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
    df_train = pd.read_parquet(str_uri, columns=list_cols_import)

    # join
    df_train = pd.merge(
        left=df_train,
        right=df_tmp,
        on='uniqueid',
        how='inner',
    )

    # get non-numeric
    list_non_numeric = []
    for col in list_cols_model:
        if df_train[col].dtype not in ['float64','int64']:
            list_non_numeric.append(col)

    # pool train
    pool_train = cb.Pool(
        df_train[list_cols_model],
        df_train['subjectage__ln'],
        list_non_numeric,
    )

    del df_train

    # pool valid
    pool_valid = cb.Pool(
        df_valid[list_cols_model],
        df_valid['subjectage__ln'],
        list_non_numeric,
    )
    del df_valid

    # init
    cls_model_inference = cb.CatBoostClassifier(
        task_type='CPU',
        nan_mode='Min',
        random_state=42,
        eval_metric='AUC',
        iterations=100,
        learning_rate=None, # default
    )

    # fit
    cls_model_inference.fit(
        pool_train,
        eval_set=[pool_valid],
        verbose=10,
        use_best_model=True,
        early_stopping_rounds=10,
    )

    # get feat imp
    df_feat_imp = pd.DataFrame({
        'feature': cls_model_inference.feature_names_,
        'importance': cls_model_inference.feature_importances_,
    })
    df_feat_imp.sort_values(by='importance', ascending=False, inplace=True)
    # write to s3
    str_filename = 'df_feat_imp_age.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/11b_disparate_impact/{str_filename}'
    df_feat_imp.to_csv(str_uri, index=False)

    ###############################################################################
    # UPLOAD
    ###############################################################################
    print('Uploading plots...')
    list_str_filename = [
        'plt_yhat_dist_race.png',
        'plt_yhat_dist_gender.png',
        'plt_yhat_dist_age.png',
    ]
    for str_filename in list_str_filename:
        upload_to_s3(
            str_local_path=f'{str_dirname_output}/{str_filename}', 
            str_bucket_path=f'02_pricing_pd/02_model/02_model/11b_disparate_impact/{str_filename}', 
            str_project=str_project,
        )
except: # if the df is empty after joining
    pass

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# image name
image=genxii-pd-disparate-valid

# Get the account number associated with the current IAM credentials
account=$(aws sts get-caller-identity --query Account --output text)

# did we have an error?
if [ $? -ne 0 ]
then
    exit 255
fi

# Get the region defined in the current configuration (default to us-west-2 if none defined)
region=$(aws configure get region)
region=${region:-us-west-2}

# get destination of repo
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# If the repository doesn't exist in ECR, create it.
aws ecr describe-repositories --repository-names "${image}" > /dev/null 2>&1

# if it doesnt exist...create it
if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "${image}" > /dev/null
fi

# Get the login command from ECR and execute it directly
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# Build the docker image locally with the image name and then push it to ECR
# with the full name.

# build and add tag
docker build  -t ${image} .
docker tag ${image} ${fullname}
# push to ecr
docker push ${fullname}

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
Sending build context to Docker daemon  75.78kB
Step 1/7 : FROM python:3.9
 ---> 33c1a19b1afb
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> 1bc5cea31762
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 53db16a067c4
Step 4/7 : COPY requirements.txt .
 ---> Using cache
 ---> 180e3a2e298e
Step 5/7 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> c5672b7010c2
Step 6/7 : COPY script.py .
 ---> 25515c1b8c32
Step 7/7 : CMD ["python3", "script.py"]
 ---> Running in f969f03bb902
Removing intermediate container f969f03bb902
 ---> 3b334c317632
Successfully built 3b334c317632
Successfully tagged genxii-pd-disparate-valid:latest
The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-disparate-valid]
1fc6b5f0260a: Preparing
bc72a217c1cd: Preparing
93761b1f50ed: Preparing
1417b313610d: Preparing
04ac96ebffcd: Preparing
b343d97c2c3c: Preparing
70981c1da3c1: Preparing
6a4ba3269682: Preparing
d3de4ba9f72c: Preparing
0c2d

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass